In [1]:
import pandas as pd
import os
from pathlib import Path
from pyecharts import options as opts
from pyecharts.charts import Pie, Grid
from pyecharts.globals import ThemeType

# 当前 python 文件所在目录
BASE_DIR = Path.cwd()

excel_list = sorted([
    f for f in BASE_DIR.glob("*.xls*")
    if not f.name.startswith("~$")
])

excel_file = pd.read_excel(excel_list[0])




In [2]:
import pandas as pd
import os
from pathlib import Path
from pyecharts import options as opts
from pyecharts.charts import Pie, Grid
from pyecharts.globals import ThemeType

# 当前 python 文件所在目录
BASE_DIR = Path.cwd()

excel_list = sorted([
    f for f in BASE_DIR.glob("*.xls*")
    if not f.name.startswith("~$")
])

excel_file = pd.read_excel(excel_list[0])




In [3]:
# 2. 数据聚合

df = excel_file

project_counts_raw = df['开发类别'].value_counts()
project_counts = list(project_counts_raw.items())

status_counts_raw = df['状态'].value_counts()
status_counts = list(status_counts_raw.items())

print(project_counts,status_counts)




[('软件开发', 19), ('MES改造', 9), ('视觉改造', 3), ('方案Review', 1), ('iPhone扫码', 1), ('运控改造', 1)] [('Done', 30), ('On-going', 4)]


In [4]:
# label color
project_counts_label_map = {
    "软件开发": "#61F0EC",
    "MES改造": "#E0A6E5",
    "视觉改造": "#32cbb6",
}

status_counts_label_map = {
    "On-going": "#FFFF00",
    "Done": "#C2B9B7",
    "Pause": "#D54660",
    "Delay": "#DA435C",
}


def create_styled_pie_data(data_source, color_map):
    final_data_pair = []
    for i, (name, val) in enumerate(data_source):
        # 轮询获取颜色
        curr_color = color_map.get(name, "#ccc")
        
        # 创建带样式的单一数据项
        item = opts.PieItem(
            name=name,
            value=val,
            # A. 设置扇区样式 (颜色 + 圆角)
            itemstyle_opts=opts.ItemStyleOpts(
                color=curr_color,        # <--- 扇区颜色
                border_radius=15, 
                border_color="#fff", 
                border_width=2
            ),
            # B. 设置标签样式 (颜色强制等于扇区颜色)
            label_opts=opts.LabelOpts(
                is_show=True,
                position="outside",
                formatter="{b} {c}\n {d}%", # 你的格式
                font_size=32,               # 你的大字号
                font_weight="bold",
                color=curr_color            # <--- 关键：标签颜色 = 扇区颜色
            )
        )
        final_data_pair.append(item)
    return final_data_pair

formatted_project_data = create_styled_pie_data(project_counts, project_counts_label_map) 
formatted_status_data = create_styled_pie_data(status_counts, status_counts_label_map) 





In [5]:
# ==========================================
# 3. 创建图表
# ==========================================
pie_project = (
    Pie(init_opts=opts.InitOpts(width="1200px", height="800px", bg_color="transparent"))
    .add(
        data_pair=formatted_project_data,
        series_name = "项目统计",
        radius=["20%", "60%"],  # 稍微改小半径，给标签腾位置
        center=["50%", "55%"], 
        rosetype="radius",
        
        # ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
        # 核心修改：直接用字典（{}）代替 opts.LabelLineOpts
        # 这样可以绕过报错，效果是一样的
        # ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼

        label_opts={
            "show": True,
            "position": "outside",
            "fontWeight": "bold",    # 【核心】加粗
            "fontSize": 18,          # 建议字号也稍微加大，更清晰
            "formatter": "{b}: {c}" 
         },# 这里的格式可以自定义

        label_line_opts={
            "show": True,
            "length": 50,    # 第一段线长度
            "length2": 100,  # 第二段线长度（文字推得更远）
            "smooth": False
        }
    )
    .set_global_opts(


        title_opts=opts.TitleOpts(
            title="SW项目统计",    # 你可以把这里改成你想显示的任何文字
            pos_left="center",
            pos_top="20",        # 离顶部 20px
            title_textstyle_opts=opts.TextStyleOpts(
                font_size=50,    # 之前的大字号
                color="#333"
            )
        ),
        legend_opts=opts.LegendOpts(is_show=False)
    )
)
pie_project.render("Chart_Project.html")
print("文件1生成完毕：Chart_Project.html")

pie_status = (
    Pie(init_opts=opts.InitOpts(width="1200px", height="800px", bg_color="transparent"))
    .add(
        data_pair=formatted_status_data,
        series_name = "项目统计",
        radius=["20%", "60%"],  # 稍微改小半径，给标签腾位置
        center=["50%", "55%"], 
        rosetype="radius",
        
        # ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
        # 核心修改：直接用字典（{}）代替 opts.LabelLineOpts
        # 这样可以绕过报错，效果是一样的
        # ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
        label_line_opts={
            "show": True,
            "length": 50,    # 第一段线长度
            "length2": 100,  # 第二段线长度（文字推得更远）
            "smooth": False
        }
    )
    .set_global_opts(


        title_opts=opts.TitleOpts(
            title="SW项目状态统计",    # 你可以把这里改成你想显示的任何文字
            pos_left="center",
            pos_top="20",        # 离顶部 20px
            title_textstyle_opts=opts.TextStyleOpts(
                font_size=50,    # 之前的大字号
                color="#333"
            )
        ),
        legend_opts=opts.LegendOpts(is_show=False)
    )
)
pie_status.render("Chart_Status.html")
print("文件1生成完毕：Chart_Status.html")

文件1生成完毕：Chart_Project.html
文件1生成完毕：Chart_Status.html
